In [1]:
import sys
sys.path.append('../../shared')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.backends.backend_pdf import PdfPages
import glob
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif'
})

df = pd.read_csv('../outputs/telco_cleaned.csv')
os.makedirs('../reports', exist_ok=True)

total      = len(df)
churned    = int(df['Churn_Binary'].sum())
retained   = total - churned
churn_rate = df['Churn_Binary'].mean() * 100

print(f'Loaded {total:,} customers  |  Churn rate: {churn_rate:.1f}%  ✓')

Loaded 7,043 customers  |  Churn rate: 26.5%  ✓


In [2]:
# ── Compute insight values ────────────────────────────────────────────────────
contract_churn = df.groupby('Contract')['Churn_Binary'].mean() * 100
mtm_rate       = contract_churn.get('Month-to-month', 0)
one_yr_rate    = contract_churn.get('One year', 0)
two_yr_rate    = contract_churn.get('Two year', 0)

internet_churn = df.groupby('InternetService')['Churn_Binary'].mean() * 100
fiber_rate     = internet_churn.get('Fiber optic', 0)
dsl_rate       = internet_churn.get('DSL', 0)

payment_churn  = df.groupby('PaymentMethod')['Churn_Binary'].mean() * 100
echeck_rate    = payment_churn.get('Electronic check', 0)

senior_rate    = df[df['SeniorCitizen'] == 1]['Churn_Binary'].mean() * 100
nonsenior_rate = df[df['SeniorCitizen'] == 0]['Churn_Binary'].mean() * 100

churned_avg_charge  = df[df['Churn']=='Yes']['MonthlyCharges'].mean()
retained_avg_charge = df[df['Churn']=='No']['MonthlyCharges'].mean()

security_no  = df[df['OnlineSecurity']=='No']['Churn_Binary'].mean() * 100
security_yes = df[df['OnlineSecurity']=='Yes']['Churn_Binary'].mean() * 100

male_rate   = df[df['gender']=='Male']['Churn_Binary'].mean() * 100
female_rate = df[df['gender']=='Female']['Churn_Binary'].mean() * 100

print('All insight values computed ✓')

All insight values computed ✓


In [3]:
# ── Print insight report ──────────────────────────────────────────────────────
DIVIDER = '=' * 62
THIN    = '-' * 62

print(DIVIDER)
print('   CUSTOMER CHURN ANALYSIS — INSIGHT REPORT')
print('   Telco Dataset  |  customer-churn-analysis / eda-insights')
print(DIVIDER)

print(f'''
DATASET OVERVIEW
{THIN}
  Total customers   : {total:,}
  Churned           : {churned:,}  ({churn_rate:.1f}%)
  Retained          : {retained:,}  ({100-churn_rate:.1f}%)
''')

insights = [
    ('1. Contract type',
     f'Month-to-month churn: {mtm_rate:.1f}%  |  '
     f'1-year: {one_yr_rate:.1f}%  |  2-year: {two_yr_rate:.1f}%\n'
     f'     Customers on short contracts are {mtm_rate/max(two_yr_rate,1):.0f}× more likely to churn.'),

    ('2. Tenure',
     'New customers (0-12 months) churn most. Loyalty stabilises\n'
     '     sharply after 2 years — early engagement is critical.'),

    ('3. Monthly charges',
     f'Churned avg: ${churned_avg_charge:.2f}/mo  |  '
     f'Retained avg: ${retained_avg_charge:.2f}/mo\n'
     f'     Higher-paying customers are at greater churn risk.'),

    ('4. Internet service',
     f'Fiber optic: {fiber_rate:.1f}% churn  |  DSL: {dsl_rate:.1f}% churn\n'
     f'     Fiber customers likely have more competitive alternatives.'),

    ('5. Payment method',
     f'Electronic check: {echeck_rate:.1f}% churn  vs  ~15-18% for auto-pay methods.\n'
     f'     Manual payers are less committed / more price-sensitive.'),

    ('6. Senior citizens',
     f'Senior churn: {senior_rate:.1f}%  |  Non-senior churn: {nonsenior_rate:.1f}%\n'
     f'     Senior customers need dedicated retention support.'),

    ('7. Online security',
     f'Without security: {security_no:.1f}% churn  |  With security: {security_yes:.1f}% churn\n'
     f'     Value-added services act as a churn buffer.'),

    ('8. Gender',
     f'Male: {male_rate:.1f}%  |  Female: {female_rate:.1f}%\n'
     f'     Gender has negligible impact on churn — not a useful segment.'),
]

print('KEY FINDINGS')
print(THIN)
for topic, detail in insights:
    print(f'\n  {topic}')
    print(f'  → {detail}')

print(f'\n{DIVIDER}')
print('  TOP 3 ACTIONABLE RECOMMENDATIONS')
print(DIVIDER)
recs = [
    ('Convert month-to-month customers',
     f'Offer discounts for annual contracts — could reduce churn from {mtm_rate:.0f}% to ~{one_yr_rate:.0f}%'),
    ('Focus on early-tenure retention',
     'Onboarding programs for 0-6 month customers have highest ROI'),
    ('Promote auto-payment',
     f'Electronic check payers churn at {echeck_rate:.0f}% — nudge toward bank transfer/credit card'),
]
for i, (title, detail) in enumerate(recs, 1):
    print(f'\n  {i}. {title}')
    print(f'     {detail}')
print(f'\n{DIVIDER}')

   CUSTOMER CHURN ANALYSIS — INSIGHT REPORT
   Telco Dataset  |  customer-churn-analysis / eda-insights

DATASET OVERVIEW
--------------------------------------------------------------
  Total customers   : 7,043
  Churned           : 1,869  (26.5%)
  Retained          : 5,174  (73.5%)

KEY FINDINGS
--------------------------------------------------------------

  1. Contract type
  → Month-to-month churn: 42.7%  |  1-year: 11.3%  |  2-year: 2.8%
     Customers on short contracts are 15× more likely to churn.

  2. Tenure
  → New customers (0-12 months) churn most. Loyalty stabilises
     sharply after 2 years — early engagement is critical.

  3. Monthly charges
  → Churned avg: $74.44/mo  |  Retained avg: $61.27/mo
     Higher-paying customers are at greater churn risk.

  4. Internet service
  → Fiber optic: 41.9% churn  |  DSL: 19.0% churn
     Fiber customers likely have more competitive alternatives.

  5. Payment method
  → Electronic check: 45.3% churn  vs  ~15-18% for auto-pay

In [4]:
# ── Export PDF report ─────────────────────────────────────────────────────────
chart_files = sorted(glob.glob('../outputs/chart_0*.png'))
print(f'Charts found: {len(chart_files)}')

PDF_PATH = '../reports/churn_analysis_report.pdf'

with PdfPages(PDF_PATH) as pdf:

    # ── Cover page ────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(8.5, 11))
    fig.patch.set_facecolor('white')

    fig.text(0.5, 0.68, 'Customer Churn Analysis',
             ha='center', fontsize=26, fontweight='bold', color='#1a1a1a')
    fig.text(0.5, 0.62, 'Telco Dataset — Exploratory Data Analysis Report',
             ha='center', fontsize=14, color='#555555')

    fig.text(0.5, 0.54, '─' * 50, ha='center', color='#cccccc', fontsize=10)

    stats = [
        ('Total customers', f'{total:,}'),
        ('Overall churn rate', f'{churn_rate:.1f}%'),
        ('Highest-risk segment', 'Month-to-month contracts'),
        ('Charts included', str(len(chart_files))),
    ]
    for i, (label, value) in enumerate(stats):
        y = 0.50 - i * 0.055
        fig.text(0.35, y, label + ':', ha='right', fontsize=12, color='#777777')
        fig.text(0.37, y, value,       ha='left',  fontsize=12, fontweight='bold', color='#1a1a1a')

    fig.text(0.5, 0.20, 'GitHub: customer-churn-analysis / eda-insights',
             ha='center', fontsize=10, color='#aaaaaa')

    pdf.savefig(fig, bbox_inches='tight')
    plt.close()

    # ── Chart pages ───────────────────────────────────────────────────────────
    chart_titles = [
        'Fig 1 — Overall Churn Split',
        'Fig 2 — Churn Rate by Contract Type',
        'Fig 3 — Churn Rate by Tenure Group',
        'Fig 4 — Monthly Charges Distribution',
        'Fig 5 — Internet Service & Payment Method',
        'Fig 6 — Demographics',
        'Fig 7 — Correlation Heatmap',
        'Fig 8 — Value-Added Services',
    ]
    for f, title in zip(chart_files, chart_titles):
        img = plt.imread(f)
        fig, ax = plt.subplots(figsize=(10, 6.5))
        fig.patch.set_facecolor('white')
        ax.imshow(img)
        ax.axis('off')
        fig.text(0.5, 0.02, title, ha='center', fontsize=9, color='#aaaaaa')
        pdf.savefig(fig, bbox_inches='tight')
        plt.close()

    # ── Summary page ──────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(8.5, 11))
    fig.patch.set_facecolor('white')
    fig.text(0.5, 0.93, 'Key Insights & Recommendations',
             ha='center', fontsize=16, fontweight='bold', color='#1a1a1a')

    summary_lines = [
        ('Contract type',  f'Month-to-month churn {mtm_rate:.0f}%  vs  2-year churn {two_yr_rate:.0f}%'),
        ('Tenure',         'New customers (0-12 mo) are highest-risk group'),
        ('Monthly charges',f'Churned customers pay ${churned_avg_charge:.0f}/mo avg vs ${retained_avg_charge:.0f}/mo'),
        ('Internet',       f'Fiber optic {fiber_rate:.0f}%  vs  DSL {dsl_rate:.0f}% churn'),
        ('Payment',        f'Electronic check {echeck_rate:.0f}% churn — highest of all methods'),
        ('Senior citizens',f'{senior_rate:.0f}% churn rate vs {nonsenior_rate:.0f}% non-senior'),
        ('Online security','Customers without it churn 2× more'),
        ('Gender',         'No meaningful impact on churn'),
    ]
    for i, (label, detail) in enumerate(summary_lines):
        y = 0.86 - i * 0.07
        fig.text(0.08, y, f'{i+1}.  {label}:', fontsize=11, fontweight='bold', color='#333333')
        fig.text(0.08, y - 0.025, f'    {detail}', fontsize=10, color='#555555')

    fig.text(0.5, 0.14, 'Top Recommendations', ha='center', fontsize=13, fontweight='bold', color='#1a1a1a')
    rec_lines = [
        f'1.  Incentivise annual contracts — month-to-month churn is {mtm_rate:.0f}%',
        '2.  Launch early-tenure onboarding (0-6 months) — highest ROI window',
        f'3.  Nudge electronic check payers to auto-pay — {echeck_rate:.0f}% vs ~16% churn',
    ]
    for i, line in enumerate(rec_lines):
        fig.text(0.08, 0.09 - i * 0.038, line, fontsize=10, color='#333333')

    pdf.savefig(fig, bbox_inches='tight')
    plt.close()

print(f'\nPDF report saved → {PDF_PATH}')
print(f'Pages: 1 cover + {len(chart_files)} charts + 1 summary = {len(chart_files)+2} total')
print('\nPath A complete ✓  —  eda-insights/')

Charts found: 4

PDF report saved → ../reports/churn_analysis_report.pdf
Pages: 1 cover + 4 charts + 1 summary = 6 total

Path A complete ✓  —  eda-insights/
